# 04 - Feature Engineering

**Goal:** Create more informative features using mechanical engineering knowledge.
**Input:** `data/processed/cnc_clean.csv`  |  **Output:** `data/processed/cnc_features.csv`

## New features
| Feature | Formula | Engineering idea |
|---|---|---|
| `cutting_intensity` | speed × feed / 1000 | Overall cutting load |
| `wear_x_vibration` | tool wear × vibration | Interaction of two major risk factors |
| `heat_per_coolant` | temperature / coolant flow | Heat generated vs heat removed |
| `vibration_per_speed` | vibration / speed × 1000 | Machine instability relative to speed |

## Encoding
`material_type` was one-hot encoded (`drop_first=True`) to avoid the dummy variable trap. `part_id` was dropped because it is only an identifier.

**Result:** `wear_x_vibration` showed the highest correlation with defects.
**Final shape:** 2,000 rows × 13 columns (12 features + target).

**Leakage check:** All new features are calculated row by row, so no test information leaks into training.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cnc_clean.csv")
print(df.shape)
df.head()

In [ ]:
fe = df.copy()

fe["cutting_intensity"] = fe["spindle_speed_rpm"] * fe["feed_rate_mm_min"] / 1000
fe["wear_x_vibration"] = fe["tool_wear_min"] * fe["vibration_mm_s"]
fe["heat_per_coolant"] = fe["cutting_temp_c"] / fe["coolant_flow_lpm"]
fe["vibration_per_speed"] = fe["vibration_mm_s"] / fe["spindle_speed_rpm"] * 1000
fe[["cutting_intensity", "wear_x_vibration","heat_per_coolant", "vibration_per_speed"]].describe().round(2)

In [ ]:
cols = ["tool_wear_min", "vibration_mm_s", "cutting_temp_c",
        "cutting_intensity", "wear_x_vibration",
        "heat_per_coolant", "vibration_per_speed"]

fe[cols + ["defect"]].corr()["defect"].drop("defect").sort_values(ascending=False).round(3)

In [ ]:
fe = pd.get_dummies(fe, columns=["material_type"], drop_first=True, dtype=int)
fe.head()

In [ ]:
fe = fe.drop(columns=["part_id"])    

print("Final shape:", fe.shape)
print(fe.columns.tolist())

fe.to_csv("../data/processed/cnc_features.csv", index=False)
print("Saved ✅")